# ML-10 — Content Action Playbook

Lane 2 is **Refresh / Content Opportunity Scoring**. This playbook is a practical, non-production ranking aid built from March 2026 warehouse data. It uses anonymized client/page identifiers and the transparent Week 4 baseline rule. The **full action queue** uses all pages with usable March 1–15 early-period signals; March 16–31 availability is used only to define a separate retrospective analytical subset. The output supports human review; it does not predict a future ranking outcome, prove refresh benefit, or authorize publication, deletion, or redirect actions.

## 1. Ranked actions + reason codes

**Queue source and population.** The queue uses the **unchanged Week 4 baseline** rather than Week 5/6 model probabilities. Features and ranking use only March 1–15 information. It is generated from all **151,981 early-period candidate pages**, regardless of whether March 16–31 late-period data is available. March 16–31 is not used for queue membership.

The exact rule adds one point for each condition: `early_impressions >= 500`, `0 < weighted_avg_position <= 20`, and `early_ctr < 0.5%`. A score of 3 receives `review_ctr_refresh`; all other scores receive `monitor`. Ranking is deterministic: descending score, descending active-search days, descending early impressions, then ascending client and content pseudonyms for ties.

Each row receives one clear baseline reason code plus a separate archetype label. `three_signal_lane2_review` means the three rule conditions are met. Other reason codes are monitor archetypes based on observable early-period evidence, not model predictions.

The **85,565-page complete-label subset** is retained separately for historical association analysis and validation context. It is not a membership filter for the action queue.

In [1]:
from pathlib import Path
import os
import json
import duckdb
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Repository root not found')

repo_root = find_repo_root(Path.cwd().resolve())
token = os.getenv('HF_TOKEN')
env_path = next((base / '.env' for base in [repo_root, *repo_root.parents] if (base / '.env').exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))
print('Local Hugging Face authentication configured:', bool(token))

march_url = hf_hub_download(
    repo_id='FlyRank/internship-warehouse', repo_type='dataset',
    filename='fact_content_daily_performance/month=2026-03/data_0.parquet', token=token,
)
con = duckdb.connect()

# Full actionable candidate population: define and aggregate only March 1-15 early information.
feature_query = '''
WITH page_windows AS (
  SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS early_impressions,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS early_clicks,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_avg_position*gsc_impressions ELSE 0 END)
      / NULLIF(SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END),0) AS weighted_avg_position,
    COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS active_search_days,
    SUM(CASE WHEN report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS late_impressions,
    COUNT(DISTINCT CASE WHEN report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS late_available_days
  FROM read_parquet(?)
  GROUP BY client_hash_id, content_hash_id
), early_candidates AS (
  SELECT *, CASE WHEN early_impressions > 0 THEN 100.0*early_clicks/early_impressions ELSE NULL END AS early_ctr
  FROM page_windows WHERE early_impressions > 0
)
SELECT client_hash_id, content_hash_id, early_impressions, early_clicks,
       weighted_avg_position, active_search_days, early_ctr,
       late_available_days,
       CASE WHEN late_available_days = 16 AND weighted_avg_position IS NOT NULL AND early_ctr IS NOT NULL
            THEN late_impressions/16.0 < early_impressions/15.0 ELSE NULL END AS is_declining_proxy
FROM early_candidates
'''
full_df = con.execute(feature_query,[march_url]).fetchdf()

population_query = '''
WITH page_windows AS (
  SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS early_impressions,
    COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS early_available_days,
    COUNT(DISTINCT CASE WHEN report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS late_available_days
  FROM read_parquet(?) GROUP BY client_hash_id, content_hash_id
)
SELECT
  COUNT(*) FILTER (WHERE early_impressions > 0 AND early_available_days > 0) AS early_candidate_pages,
  COUNT(*) FILTER (WHERE early_impressions > 0 AND early_available_days > 0 AND late_available_days = 16) AS evaluable_later_label_pages,
  COUNT(*) FILTER (WHERE early_impressions > 0 AND early_available_days > 0 AND late_available_days >= 1 AND late_available_days < 16) AS partial_late_availability_pages
FROM page_windows
'''
early_candidates, evaluable_candidates, partial_late = con.execute(population_query,[march_url]).fetchone()

# Retrospective subset only; the proxy is not used for queue membership.
evaluable_df = full_df[(full_df['late_available_days'] == 16) & full_df['is_declining_proxy'].notna()].copy()
evaluable_df['is_declining_proxy'] = evaluable_df['is_declining_proxy'].astype(int)
df = full_df  # Backward-compatible dataframe used by the baseline queue.

print('Full early-period action queue candidates:', len(full_df))
print('Early-period candidate population from March 1-15 query:', int(early_candidates))
print('Historical evaluable March 16-31 subset:', len(evaluable_df))
print('Historical evaluable subset reported by population query:', int(evaluable_candidates))
print('Pages with partial late availability:', int(partial_late))
print('Queue membership uses no March 16-31 availability or target condition.')

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Local Hugging Face authentication configured: True


Full early-period action queue candidates: 151981
Early-period candidate population from March 1-15 query: 151981
Historical evaluable March 16-31 subset: 85565
Historical evaluable subset reported by population query: 85565
Pages with partial late availability: 55902
Queue membership uses no March 16-31 availability or target condition.


In [2]:
# Reproduce the exact unchanged Week 4 baseline scoring rule on ALL early-period candidates.
queue = df.copy()
queue['search_demand_met'] = (queue['early_impressions'] >= 500).astype(int)
queue['position_opportunity_met'] = ((queue['weighted_avg_position'] > 0) & (queue['weighted_avg_position'] <= 20)).astype(int)
queue['low_ctr_met'] = (queue['early_ctr'] < 0.5).astype(int)
queue['baseline_refresh_score'] = queue['search_demand_met'] + queue['position_opportunity_met'] + queue['low_ctr_met']

queue['reason_code'] = queue['baseline_refresh_score'].apply(
    lambda s: 'three_signal_lane2_review' if s == 3 else ('two_signal_monitor' if s == 2 else ('one_signal_monitor' if s == 1 else 'insufficient_evidence_monitor'))
)
queue['review_archetype'] = 'weak_or_insufficient_evidence'
queue.loc[(queue['baseline_refresh_score'] == 3), 'review_archetype'] = 'high_impressions_low_ctr_visible'
queue.loc[(queue['baseline_refresh_score'] == 2) & (queue['early_impressions'] >= 500) & (queue['weighted_avg_position'] <= 20), 'review_archetype'] = 'visible_demand_near_opportunity'
queue.loc[(queue['baseline_refresh_score'] < 2), 'review_archetype'] = 'weak_or_insufficient_evidence'

queue['suggested_action'] = queue['baseline_refresh_score'].apply(
    lambda s: 'review_ctr_refresh' if s == 3 else 'monitor'
)
queue['confidence_limitations'] = (
    'March 1-15 early-period signal only; historical proxy is not included or used for ranking; no refresh-benefit evidence.'
)
queue = queue.sort_values(
    ['baseline_refresh_score','active_search_days','early_impressions','client_hash_id','content_hash_id'],
    ascending=[False,False,False,True,True], kind='mergesort'
).reset_index(drop=True)
queue['rank'] = np.arange(1, len(queue)+1)

ranked_queue = queue[['rank','client_hash_id','content_hash_id','baseline_refresh_score','reason_code','review_archetype','suggested_action','confidence_limitations','early_impressions','early_clicks','weighted_avg_position','active_search_days','early_ctr']].copy()

out_dir = repo_root/'work'/'outputs'
fig_dir = repo_root/'work'/'figures'
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
queue_path = out_dir/'w07_lane2_action_queue.csv'
ranked_queue.to_csv(queue_path,index=False)

print('Full early-period queue rows:', len(ranked_queue))
print('Action distribution:')
print(ranked_queue['suggested_action'].value_counts().to_string())
print('Reason codes:')
print(ranked_queue['reason_code'].value_counts().to_string())
print('Archetypes:')
print(ranked_queue['review_archetype'].value_counts().to_string())
print('Historical evaluable subset rows:', len(evaluable_df))
print('Queue CSV:', queue_path.relative_to(repo_root).as_posix())
assert len(ranked_queue) == 151981
assert len(ranked_queue) == int(early_candidates)
assert len(evaluable_df) == 85565
assert not ranked_queue['reason_code'].isna().any()
assert not ranked_queue['suggested_action'].isna().any()
assert ranked_queue['baseline_refresh_score'].between(0,3).all()
assert ranked_queue['rank'].is_monotonic_increasing

Full early-period queue rows: 151981
Action distribution:
suggested_action
monitor               125384
review_ctr_refresh     26597
Reason codes:
reason_code
two_signal_monitor               83471
one_signal_monitor               40307
three_signal_lane2_review        26597
insufficient_evidence_monitor     1606
Archetypes:
review_archetype
weak_or_insufficient_evidence       117772
high_impressions_low_ctr_visible     26597
visible_demand_near_opportunity       7612
Historical evaluable subset rows: 85565
Queue CSV: work/outputs/w07_lane2_action_queue.csv


### Archetype-to-action mapping

The mapping is intentionally simple and uses only supported columns:

- **High impressions + low CTR + visible position:** `three_signal_lane2_review` and `high_impressions_low_ctr_visible`; human-review `review_ctr_refresh` for title/snippet, intent alignment, SERP context, freshness, and competing pages. This archetype is exactly score 3.
- **Visible demand and two signals:** `two_signal_monitor` / `visible_demand_near_opportunity`; suggested action remains `monitor`, with review possible if context justifies it. It is lower priority than score 3.
- **Strong position/CTR or otherwise weak combinations:** `one_signal_monitor` or `insufficient_evidence_monitor`; protect high-quality assets and monitor weak evidence rather than recommend a major change.
- **Stale-old-page archetype:** not included because this warehouse build does not provide a validated days-since-update field in the Week 4/5/6 feature design.

This is a prioritization aid. It does not state that content is stale, broken, or causally declining.

## 2. Intended use and limits

Content owners, SEOs, or analysts may use the queue to allocate limited review time. It supports ordering work for contextual inspection and comparing pages with similar early-period signals. It does not establish causality, a future ranking forecast, a refresh benefit, ranking guarantees, deletion suitability, or a recommendation to publish changes.

**Population distinction.** The actionable queue contains all 151,981 March 1–15 candidate pages. A separate 85,565-page historical subset has a complete March 16–31 proxy label and is used only for retrospective association/validation context. The model or queue score must not be described as producing outcomes for the 85,565 subset alone.

**Measured limitation.** The Week 6 grouped holdout measured 4,113 test pages from 8 test clients, with 31 training clients and zero client overlap, Precision@20 = 0.5500, Precision@50 = 0.4800, ROC-AUC = 0.7667, and base rate = 0.3221. The small test-client count limits confidence. Those grouped metrics describe the logistic model on its own evaluable historical sample; they do not describe the full Week 7 baseline queue.

The proxy compares March 16–31 daily late impressions with March 1–15 daily early impressions. It may reflect seasonality, temporary demand, algorithm changes, indexing, competitor activity, measurement gaps, and content-quality effects. The queue is therefore a decision-support list, not an autonomous decision-maker.

In [3]:
# Recompute and print the validated grouped-model comparison for honest reporting only.
feature_columns = ['early_impressions','early_clicks','weighted_avg_position','active_search_days','early_ctr']
validation_df = evaluable_df.copy()
X = validation_df[feature_columns].copy()
y = validation_df['is_declining_proxy'].astype(int)
groups = validation_df['client_hash_id'].astype(str)
gss = GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=42)
g_train,g_test = next(gss.split(X,y,groups=groups))
model = Pipeline([
    ('scaler',StandardScaler()),
    ('model',LogisticRegression(class_weight='balanced',max_iter=1000,random_state=42)),
])
model.fit(X.iloc[g_train],y.iloc[g_train])
heldout = validation_df.iloc[g_test].copy()
heldout['model_probability'] = model.predict_proba(X.iloc[g_test])[:,1]

def precision_at_k(frame,k):
    return float(frame.sort_values('model_probability',ascending=False,kind='mergesort').head(k)['is_declining_proxy'].mean())

week6_metrics = {
    'train_pages': int(len(g_train)), 'test_pages': int(len(g_test)),
    'train_clients': int(validation_df.iloc[g_train]['client_hash_id'].nunique()),
    'test_clients': int(heldout['client_hash_id'].nunique()),
    'client_overlap': int(len(set(validation_df.iloc[g_train]['client_hash_id']) & set(heldout['client_hash_id']))),
    'precision_at_20': precision_at_k(heldout,20), 'precision_at_50': precision_at_k(heldout,50),
    'roc_auc': float(roc_auc_score(heldout['is_declining_proxy'],heldout['model_probability'])),
    'positive_class_rate': float(heldout['is_declining_proxy'].mean()),
}
print(json.dumps(week6_metrics,indent=2))

{
  "train_pages": 81452,
  "test_pages": 4113,
  "train_clients": 31,
  "test_clients": 8,
  "client_overlap": 0,
  "precision_at_20": 0.55,
  "precision_at_50": 0.48,
  "roc_auc": 0.7667188219052002,
  "positive_class_rate": 0.32214928276197424
}


## 3. Human review + the no-go list

For every recommended row, a reviewer should inspect current visibility and search intent, page quality and relevance, title/snippet rendering, indexing status, meaningful changes since March, competitor SERPs, seasonality, and whether the page is actually stale or newly launched. The reviewer should document acceptance or rejection and the reason.

**No-go automation:**

- Do not delete or prune pages.
- Do not publish or rewrite content automatically.
- Do not redirect URLs.
- Do not change canonical tags or indexation.
- Do not make ranking or traffic guarantees.
- Do not make high-impact client decisions from the model or baseline score alone.

Reject, downgrade, or defer when the page is no longer searchable, content is already current, product or legal needs prevent a change, traffic is known to be seasonal, the signal is stale, measurement is incomplete, or the reviewer sees no safe refresh hypothesis.

In [4]:
review_rules = {
    'before_action': [
        'Check current search visibility and intent.', 'Inspect indexation and content relevance.',
        'Check title/snippet and competitor context.', 'Check current freshness and meaningful edits.',
        'Check whether the traffic signal is seasonal or measurement noise.',
    ],
    'reject_downgrade_defer': [
        'No longer visible or indexable.', 'Already current and relevant.',
        'High legal, product, or brand risk.', 'Likely temporary seasonal/competitive event.',
        'Insufficient or stale data.',
    ],
    'never_automate': [
        'delete_or_prune','publish_or_rewrite','redirect','canonical_or_indexation_changes',
        'ranking_guarantees','high_impact_client_decisions_from_score_only'
    ]
}
print(json.dumps(review_rules,indent=2))

{
  "before_action": [
    "Check current search visibility and intent.",
    "Inspect indexation and content relevance.",
    "Check title/snippet and competitor context.",
    "Check current freshness and meaningful edits.",
    "Check whether the traffic signal is seasonal or measurement noise."
  ],
  "reject_downgrade_defer": [
    "No longer visible or indexable.",
    "Already current and relevant.",
    "High legal, product, or brand risk.",
    "Likely temporary seasonal/competitive event.",
    "Insufficient or stale data."
  ],
  "never_automate": [
    "delete_or_prune",
    "publish_or_rewrite",
    "redirect",
    "canonical_or_indexation_changes",
    "ranking_guarantees",
    "high_impact_client_decisions_from_score_only"
  ]
}


## 4. Monitoring / retrain triggers

This playbook describes a light, non-production workflow:

- Track queue size and the distribution of actions and archetypes.
- Record human acceptance/rejection rates and the reason a reviewer rejected or deferred a page.
- When future labels become available, calculate Precision@K on subsequently observed outcomes and compare against the documented 20/50 capacities.
- Monitor missingness, data-availability days, feature drift, and late-period label availability.
- Monitor changes in the positive-class/base rate and the distribution of impressions, CTR, active days, and position.
- Track model or baseline ranking stability across reruns and client partitions.

The following proposed thresholds are operating ideas, not measured production facts: initiate a documented retraining/review if acceptance falls below 50% for 20 reviewed pages, Precision@20 drops by 10 percentage points relative to the grouped holdout, missingness exceeds 10% for a feature, or the positive base rate changes by 10 percentage points. Stability or data-change concerns may also trigger review even when no numerical threshold is crossed.

In [5]:
monitoring_plan = {
    'queue_size_and_distribution':'Track size and action/archetype counts each run.',
    'human_acceptance_rejection':'Record reviewer decision and reason for every reviewed page.',
    'precision_at_k':'Compute on later observed labels only; no invented outcomes.',
    'feature_drift':'Compare missingness and distributions for the five March 1-15 features.',
    'base_rate_drift':'Track proxy base-rate changes when subsequent labels are available.',
    'ranking_stability':'Measure rerank stability and client-level concentration.',
    'proposed_retrain_triggers':{
        'acceptance_rate_below_0.50':'After at least 20 reviewed pages; proposed only.',
        'precision_at_20_drop_10_points':'Relative to the grouped Week 6 holdout; proposed only.',
        'feature_missingness_above_0.10':'Proposed data-quality operating threshold.',
        'base_rate_change_10_points':'Proposed monitoring threshold.',
    }
}
print(json.dumps(monitoring_plan,indent=2))

{
  "queue_size_and_distribution": "Track size and action/archetype counts each run.",
  "human_acceptance_rejection": "Record reviewer decision and reason for every reviewed page.",
  "precision_at_k": "Compute on later observed labels only; no invented outcomes.",
  "feature_drift": "Compare missingness and distributions for the five March 1-15 features.",
  "base_rate_drift": "Track proxy base-rate changes when subsequent labels are available.",
  "ranking_stability": "Measure rerank stability and client-level concentration.",
  "proposed_retrain_triggers": {
    "acceptance_rate_below_0.50": "After at least 20 reviewed pages; proposed only.",
    "precision_at_20_drop_10_points": "Relative to the grouped Week 6 holdout; proposed only.",
    "feature_missingness_above_0.10": "Proposed data-quality operating threshold.",
    "base_rate_change_10_points": "Proposed monitoring threshold."
  }
}


## 5. Cost/value thinking

This section uses qualitative assumptions because no monetary review, refresh, or expansion costs were provided.

- **Human review:** usually lower risk and moderate staff cost; it can reject weak recommendations and inspect context.
- **Refreshing a page:** higher implementation and editorial cost than review, with uncertain payoff; do only after evidence and content/intent review.
- **Expanding content:** potentially highest research and production cost; justified only where missing relevance or depth can be verified.
- **Protecting a strong page:** value is avoiding accidental harm, with relatively low incremental monitoring/review cost.
- **Monitoring only:** lowest intervention cost, but it may defer opportunity and needs reviewer-defined follow-up.
- **Pruning or redirecting:** high risk and high rollback/relevance cost; it should require stronger evidence and governance than other actions.

Assumption: review is safer and cheaper than implementing an action, and no page should receive a high-cost intervention based only on a score. These are operating assumptions, not measured costs or ROI estimates.

## 6. Exports for the paper

The notebook regenerates a Git-ignored ranked queue CSV, reusable figures, and a JSON metrics receipt. The receipt separates the full 151,981-page March 1–15 action population from the 85,565-page March 16–31 historical evaluable subset. It records each population's purpose, the queue method and feature window, outcome-analysis definitions, and relevant Week 6 grouped metrics and limitations.

In [6]:
# Create actual decay/refresh association evidence and reusable figures.
position_df = evaluable_df[evaluable_df['weighted_avg_position'] > 0].copy()
position_df['position_bucket'] = pd.cut(position_df['weighted_avg_position'],bins=[0,3,10,20,float('inf')],labels=['top_3_1_3','page1_4_10','striking_11_20','deep_or_lower_gt20'],right=True)
position_df['ctr_bucket'] = np.where(position_df['early_ctr'] < 0.5,'low_ctr_<0.5_pct','other_or_at_least_0.5_pct')
ctr_table = position_df.groupby(['position_bucket','ctr_bucket'],observed=True).agg(n=('is_declining_proxy','size'),declining_proxy_rate=('is_declining_proxy','mean')).reset_index()
print('Low-CTR decay-proxy association by position:')
print(ctr_table.to_string(index=False,float_format=lambda x:f'{x:.4f}'))

low = ctr_table[ctr_table['ctr_bucket']=='low_ctr_<0.5_pct'].copy()
high = ctr_table[ctr_table['ctr_bucket']=='other_or_at_least_0.5_pct'].copy()

volume_df = evaluable_df.copy()
volume_df['volume_bucket'] = pd.cut(volume_df['early_impressions'],bins=[-1,499,2999,float('inf')],labels=['low_or_none_<500','moderate_500_2999','high_>=3000'],right=True)
volume_table = volume_df.groupby('volume_bucket',observed=True).agg(n=('is_declining_proxy','size'),declining_proxy_rate=('is_declining_proxy','mean')).reset_index()
print('Search-demand association:')
print(volume_table.to_string(index=False,float_format=lambda x:f'{x:.4f}'))

# Figure 1: queue action distribution.
counts = ranked_queue['suggested_action'].value_counts().sort_index()
plot_table = ctr_table.pivot(index='position_bucket', columns='ctr_bucket', values='declining_proxy_rate')

def write_action_svg(path, counts):
    width,height=720,420
    max_count=max(counts.values)
    lines=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
           '<rect width="100%" height="100%" fill="white"/>',
           '<text x="360" y="35" text-anchor="middle" font-family="sans-serif" font-size="20">Week 7 Lane 2 Action Queue Distribution</text>']
    colors=['#6c7a89','#c0392b']
    for i,(label,count) in enumerate(counts.items()):
        bar_w=520*float(count)/float(max_count)
        y=110+i*110
        lines += [f'<text x="20" y="{y+20}" font-family="sans-serif" font-size="16">{label}</text>',
                  f'<rect x="230" y="{y}" width="{bar_w:.1f}" height="55" fill="{colors[i % len(colors)]}"/>',
                  f'<text x="{230+bar_w+10:.1f}" y="{y+34}" font-family="sans-serif" font-size="16">{count}</text>']
    lines.append('</svg>')
    path.write_text(chr(10).join(lines))

action_fig=fig_dir/'w07_action_distribution.svg'
write_action_svg(action_fig,counts)

# Figure 2: low CTR versus other CTR by position band.
def write_decay_svg(path, plot_table):
    width,height=900,520
    lines=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
           '<rect width="100%" height="100%" fill="white"/>',
           '<text x="450" y="35" text-anchor="middle" font-family="sans-serif" font-size="20">Observed Proxy Rate by Position and Early CTR</text>',
           '<text x="25" y="85" font-family="sans-serif" font-size="14">Early CTR: low=&lt;0.5% (blue), other=&gt;=0.5% (orange)</text>']
    bands=list(plot_table.index.astype(str))
    low_col=plot_table.columns[plot_table.columns.str.contains('low',case=False)][0]
    other_col=[c for c in plot_table.columns if c != low_col][0]
    for i,band in enumerate(bands):
        y=130+i*85
        lowv=float(plot_table.loc[band,low_col]); otherv=float(plot_table.loc[band,other_col])
        lines += [f'<text x="20" y="{y+18}" font-family="sans-serif" font-size="14">{band}</text>',
                  f'<rect x="180" y="{y}" width="{600*lowv:.1f}" height="25" fill="#3366cc"/>',
                  f'<text x="{180+600*lowv+8:.1f}" y="{y+18}" font-family="sans-serif" font-size="13">{lowv:.3f}</text>',
                  f'<rect x="180" y="{y+32}" width="{600*otherv:.1f}" height="25" fill="#ed7d31"/>',
                  f'<text x="{180+600*otherv+8:.1f}" y="{y+50}" font-family="sans-serif" font-size="13">{otherv:.3f}</text>']
    lines.append('</svg>')
    path.write_text(chr(10).join(lines))

decay_fig=fig_dir/'w07_decay_proxy_by_position.svg'
write_decay_svg(decay_fig,plot_table)
print('Figure exports:', decay_fig.relative_to(repo_root).as_posix(), action_fig.relative_to(repo_root).as_posix())
print('Figure exports:', action_fig.relative_to(repo_root).as_posix(), decay_fig.relative_to(repo_root).as_posix())

Low-CTR decay-proxy association by position:
   position_bucket                ctr_bucket     n  declining_proxy_rate
         top_3_1_3          low_ctr_<0.5_pct  7539                0.4396
         top_3_1_3 other_or_at_least_0.5_pct  2815                0.2785
        page1_4_10          low_ctr_<0.5_pct 31817                0.4535
        page1_4_10 other_or_at_least_0.5_pct  8848                0.3130
    striking_11_20          low_ctr_<0.5_pct 12506                0.4092
    striking_11_20 other_or_at_least_0.5_pct  2794                0.3604
deep_or_lower_gt20          low_ctr_<0.5_pct 17738                0.4400
deep_or_lower_gt20 other_or_at_least_0.5_pct  1469                0.3901
Search-demand association:
    volume_bucket     n  declining_proxy_rate
 low_or_none_<500 44212                0.3396
moderate_500_2999 31283                0.4927
      high_>=3000 10070                0.5332
Figure exports: work/figures/w07_decay_proxy_by_position.svg work/figures/w07_action_di

### Decay/refresh interpretation

Week 4 measured a **confirmed low-CTR association**: in every tested position band in that notebook, pages with March CTR below 0.5% had a higher historical decline-proxy rate than the comparison bucket. The current March evaluable data can also produce the association by position; it is directional and observational.

The search-demand signal was mixed: the high-demand bucket was higher, but demand was not a monotonic predictor and moderate demand was similar to low demand. The playbook therefore treats decay/refresh cues as review hypotheses, not causal refresh effects. Refreshing a page was not randomized and no content action was observed, so this analysis cannot claim that a refresh will recover traffic or improve rankings.

In [7]:
metrics_receipt = {
    'assignment':'w07_action_playbook',
    'lane':'Lane 2 - Refresh / Content Opportunity Scoring',
    'development_month':'2026-03',
    'queue_generation_method':{
        'source':'unchanged Week 4 transparent baseline',
        'feature_window':'March 1-15 inclusive',
        'target_window':'March 16-31, joined separately only for retrospective analysis and not used for queue membership or predictive inputs',
        'rule':[
            'early_impressions >= 500: +1',
            '0 < weighted_avg_position <= 20: +1',
            'early_ctr < 0.5%: +1',
        ],
        'action':'score 3 receives review_ctr_refresh; otherwise monitor',
        'tie_breaking':['baseline_refresh_score descending','active_search_days descending','early_impressions descending','client_hash_id ascending','content_hash_id ascending'],
    },
    'population':{
        'full_early_period_action_queue':{
            'purpose':'All eligible pages ranked for human review using March 1-15 information only',
            'candidate_pages':int(len(ranked_queue)),
            'action_distribution':ranked_queue['suggested_action'].value_counts().to_dict(),
        },
        'historical_evaluable_subset':{
            'purpose':'Retrospective decline-proxy association and outcome analysis only; not used for queue membership',
            'evaluable_later_label_pages':int(len(evaluable_df)),
            'population_query_evaluable_later_label_pages':int(evaluable_candidates),
            'partial_late_availability_pages':int(partial_late),
        },
        'early_candidate_population_query':int(early_candidates),
    },
    'week6_grouped_logistic_validation':week6_metrics,
    'validation_limits':[
        'Metrics apply to the 16-date historical evaluable subset, not to the full early-period action queue.',
        'The grouped holdout uses 8 test clients and zero client overlap, so confidence is limited.',
        'The proxy is historical association and not future decline or refresh benefit.',
        'The action queue uses March 1-15 features and the Week 4 baseline, not held-out logistic probabilities or later-period availability.',
    ],
    'exports':{
        'queue_csv':queue_path.relative_to(repo_root).as_posix(),
        'action_figure':action_fig.relative_to(repo_root).as_posix(),
        'decay_figure':decay_fig.relative_to(repo_root).as_posix(),
    },
    'public_safety':['No private names, URLs, raw queries, tokens, target-derived columns, or late-period fields are exported in the action queue.'],
}
metrics_path = out_dir/'w07_action_playbook_metrics.json'
metrics_path.write_text(json.dumps(metrics_receipt,indent=2)+chr(10))
print('Metrics JSON:', metrics_path.relative_to(repo_root).as_posix())
print(json.dumps(metrics_receipt['population'],indent=2))
assert metrics_path.exists()
assert action_fig.exists() and decay_fig.exists()
with metrics_path.open() as f:
    json.load(f)

Metrics JSON: work/outputs/w07_action_playbook_metrics.json
{
  "full_early_period_action_queue": {
    "purpose": "All eligible pages ranked for human review using March 1-15 information only",
    "candidate_pages": 151981,
    "action_distribution": {
      "monitor": 125384,
      "review_ctr_refresh": 26597
    }
  },
  "historical_evaluable_subset": {
    "purpose": "Retrospective decline-proxy association and outcome analysis only; not used for queue membership",
    "evaluable_later_label_pages": 85565,
    "population_query_evaluable_later_label_pages": 85565,
    "partial_late_availability_pages": 55902
  },
  "early_candidate_population_query": 151981
}


## 7. Self-check

- [ ] Ranked queue exists and is non-empty
- [ ] Every queue row has a reason code and suggested action
- [ ] Action mapping and archetypes are documented
- [ ] Decay/refresh insight is linked to actual measured associations
- [ ] Intended use and limitations are explicit
- [ ] Human-review rules and no-go automation are explicit
- [ ] Monitoring and retrain triggers are included
- [ ] Cost/value assumptions are disclosed
- [ ] Queue CSV is excluded by the repository `.gitignore`
- [ ] Figures and metrics JSON are generated and valid
- [ ] No private names, URLs, raw queries, or tokens are exposed
- [ ] All notebook cells execute successfully

In [8]:
checks = {
    'ranked_queue_non_empty': len(ranked_queue) > 0,
    'all_rows_have_reason_and_action': (ranked_queue['reason_code'].notna().all() and ranked_queue['suggested_action'].notna().all()),
    'action_mapping_documented': set(ranked_queue['suggested_action']).issubset({'review_ctr_refresh','monitor'}),
    'queue_is_public_safe_columns': not set(ranked_queue.columns).intersection({'url','client_name','raw_query','hf_token','late_impressions','late_clicks','is_declining_proxy','trend_direction','trend_pct'}),
    'decay_analysis_generated': 'ctr_table' in globals() and len(ctr_table) > 0,
    'figure_exports_exist': action_fig.exists() and decay_fig.exists(),
    'queue_csv_exists': queue_path.exists(),
    'metrics_json_valid': metrics_path.exists(),
}
print('Self-check results')
for name,passed in checks.items():
    print(f'- {name}: {"PASS" if passed else "FAIL"}')
assert all(checks.values())
print('All self-check assertions passed: True')

Self-check results
- ranked_queue_non_empty: PASS
- all_rows_have_reason_and_action: PASS
- action_mapping_documented: PASS
- queue_is_public_safe_columns: PASS
- decay_analysis_generated: PASS
- figure_exports_exist: PASS
- queue_csv_exists: PASS
- metrics_json_valid: PASS
All self-check assertions passed: True
